In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
url ='https://raw.githubusercontent.com/vp763/ML_Project_cognifyz/refs/heads/main/Dataset%20.csv'
data = pd.read_csv(url)

In [3]:
data.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [6]:
df = data.drop([
    'Restaurant ID', 'Address', 'Latitude', 'Longitude',
    'Rating color', 'Rating text'
], axis=1)

In [7]:
df['Has Online delivery'] = df['Has Online delivery'].map({'Yes':1, 'No':0})
df['Has Table booking'] = df['Has Table booking'].map({'Yes':1, 'No':0})
df['Is delivering now'] = df['Is delivering now'].map({'Yes':1, 'No':0})

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
df['Cuisines'] = df['Cuisines'].fillna('')
cuisine_matrix = tfidf.fit_transform(df['Cuisines'])

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(cuisine_matrix, cuisine_matrix)

In [12]:
def recommend_restaurant(name, cosine_sim=cosine_sim):

    idx = df[df['Restaurant Name'] == name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    restaurant_indices = [i[0] for i in sim_scores]

    return df[['Restaurant Name', 'Cuisines', 'Aggregate rating']].iloc[restaurant_indices]

In [13]:
recommend_restaurant("Domino's Pizza")

,Restaurant Name,Cuisines,Aggregate rating
32,Rovereto,Pizza,3.1
154,Flying Pie Pizzaria,Pizza,4.1
156,Guido's Original New York Style Pizza,Pizza,4.3
179,A & A Pagliai's Pizza,Pizza,4.3
202,Mellow Mushroom,Pizza,4.1


In [14]:
df['combined_features'] = (
    df['Cuisines'] + " " +
    df['Price range'].astype(str) + " " +
    df['Has Online delivery'].astype(str) + " " +
    df['Has Table booking'].astype(str)
)

tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(df['combined_features'])

cosine_sim = cosine_similarity(feature_matrix)

Evaluate Recommendation Quality
You can test with:

Case 1:
User wants:

Cheap restaurants
Online delivery
Indian cuisine
Filter:

In [15]:
df[(df['Cuisines'].str.contains("Indian")) &
   (df['Price range'] == 1) &
   (df['Has Online delivery'] == 1)]

,Restaurant Name,Country Code,City,Locality,Locality Verbose,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Votes,combined_features
737,Eat Street,1,Bangalore,Koramangala 6th Block,"Koramangala 6th Block, Bangalore","North Indian, Chinese, Italian, Street Food, D...",400,Indian Rupees(Rs.),0,1,0,No,1,4.3,753,"North Indian, Chinese, Italian, Street Food, D..."
916,Smugglers,1,Faridabad,Sector 10,"Sector 10, Faridabad",North Indian,450,Indian Rupees(Rs.),0,1,0,No,1,3.1,7,North Indian 1 1 0
1020,Anupama Sweets & Family Restaurant,1,Faridabad,Sector 34,"Sector 34, Faridabad","Mithai, North Indian, South Indian, Chinese, F...",400,Indian Rupees(Rs.),0,1,0,No,1,2.7,28,"Mithai, North Indian, South Indian, Chinese, F..."
1077,Anupam Sweets & Restaurant,1,Faridabad,Sector 86,"Sector 86, Faridabad","North Indian, Chinese, Fast Food",450,Indian Rupees(Rs.),0,1,0,No,1,3.0,10,"North Indian, Chinese, Fast Food 1 1 0"
1118,FoodByMom,1,Ghaziabad,Indirapuram,"Indirapuram, Ghaziabad",North Indian,300,Indian Rupees(Rs.),0,1,0,No,1,3.6,161,North Indian 1 1 0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8925,Tiwari Hot Pot,1,Noida,Sector 63,"Sector 63, Noida","North Indian, Chinese, South Indian",450,Indian Rupees(Rs.),0,1,0,No,1,2.6,11,"North Indian, Chinese, South Indian 1 1 0"
8937,Twomato Foods,1,Noida,Sector 65,"Sector 65, Noida",North Indian,400,Indian Rupees(Rs.),0,1,0,No,1,3.2,11,North Indian 1 1 0
9011,Bikaner's,1,Noida,Sector 72,"Sector 72, Noida","Mithai, North Indian",400,Indian Rupees(Rs.),0,1,0,No,1,2.4,11,"Mithai, North Indian 1 1 0"
9016,Breaktym,1,Noida,Sector 93,"Sector 93, Noida","North Indian, Mughlai",400,Indian Rupees(Rs.),0,1,0,No,1,3.2,20,"North Indian, Mughlai 1 1 0"
